In [1]:
%run "./00_config.ipynb"

JAVA_HOME: C:\Java\jdk-17
java.exe found at: C:\Java\jdk-17\bin\java.EXE
PySpark home: c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark
bin dir exists: True
['beeline', 'beeline.cmd', 'docker-image-tool.sh', 'find-spark-home', 'find-spark-home.cmd', 'load-spark-env.cmd', 'load-spark-env.sh', 'pyspark', 'pyspark.cmd', 'pyspark2.cmd', 'run-example', 'run-example.cmd', 'spark-class', 'spark-class.cmd', 'spark-class2.cmd', 'spark-connect-shell', 'spark-shell', 'spark-shell.cmd', 'spark-shell2.cmd', 'spark-sql', 'spark-sql.cmd', 'spark-sql2.cmd', 'spark-submit', 'spark-submit.cmd', 'spark-submit2.cmd', 'sparkR', 'sparkR.cmd', 'sparkR2.cmd']
SPARK_HOME env: None
SPARK_HOME: None
JAVA_HOME  : C:\Java\jdk-17
HADOOP_HOME: C:\hadoop
winutils found at: C:\hadoop\bin\winutils.exe
Ready: C:\covid_pipeline\bronze
Ready: C:\covid_pipeline\silver
Ready: C:\covid_pipeline\gold
Spark version: 3.5.3
Spark master : local[*]
Parquet write test succeeded at: C:\covid_pipeline\

In [ ]:
# type: ignore

In [2]:
import sys
print(sys.executable)

c:\Users\Asus\AppData\Local\Programs\Python\Python311\python.exe


In [3]:
import pandas as pd
from pyspark.sql import functions as F

WORLDBANK_XLSX = os.path.join(BRONZE_DIR, "world_bank_income_classification.xlsx")

pdf = pd.read_excel(WORLDBANK_XLSX, sheet_name="List of economies")
print(pdf.shape)
pdf.head()

(266, 5)


,Economy,Code,Region,Income group,Lending category
0,Afghanistan,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,IDA
1,Albania,ALB,Europe & Central Asia,Upper middle income,IBRD
2,Algeria,DZA,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,IBRD
3,American Samoa,ASM,East Asia & Pacific,High income,NaN
4,Andorra,AND,Europe & Central Asia,High income,NaN


In [4]:
pdf_clean = pdf.rename(columns={
    "Economy": "country",
    "Code": "iso_code",
    "Region": "wb_region",
    "Income group": "income_group",
    "Lending category": "lending_category",
})[["iso_code", "country", "wb_region", "income_group", "lending_category"]]

# Drop rows with no income group assigned (aggregates/footnote rows at bottom of sheet, if any)
pdf_clean = pdf_clean.dropna(subset=["iso_code", "income_group"])

print("Row count:", len(pdf_clean))
print(pdf_clean["income_group"].value_counts())

df_clean = spark.createDataFrame(pdf_clean)
df_clean.show(5)

Row count: 218
income_group
High income            87
Upper middle income    59
Lower middle income    47
Low income             25
Name: count, dtype: int64
+--------+--------------+--------------------+-------------------+----------------+
|iso_code|       country|           wb_region|       income_group|lending_category|
+--------+--------------+--------------------+-------------------+----------------+
|     AFG|   Afghanistan|Middle East, Nort...|         Low income|             IDA|
|     ALB|       Albania|Europe & Central ...|Upper middle income|            IBRD|
|     DZA|       Algeria|Middle East, Nort...|Upper middle income|            IBRD|
|     ASM|American Samoa| East Asia & Pacific|        High income|             NaN|
|     AND|       Andorra|Europe & Central ...|        High income|             NaN|
+--------+--------------+--------------------+-------------------+----------------+
only showing top 5 rows



In [5]:
df_clean.write.mode("overwrite").parquet(SILVER_WORLDBANK)
print("Written to:", SILVER_WORLDBANK)

Written to: C:\covid_pipeline\silver\worldbank
